In [1]:
import os
import scanpy as sc
import pandas as pd
import numpy as np

In [2]:
adata = sc.read_h5ad("Stage18_Data.h5ad")

/var/folders/k3/vg_c2k5x4mbcx6ktwfvjsddw0000gn/T/ipykernel_97776/3296280373.py:1: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  adata = sc.read_h5ad("Stage18_Data.h5ad")


In [3]:
print(adata)


AnnData object with n_obs × n_vars = 30508 × 22806
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.3.9', 'seurat_clusters'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances'
    layers: None (.X)


In [14]:
print(adata.obs['orig.ident'].value_counts())


orig.ident
pax6_Mutant    8582
six3_WT        8028
six3_Mutant    7499
pax6_WT        6399
Name: count, dtype: int64


In [15]:
adata_wt = adata[adata.obs['orig.ident'] == 'pax6_WT'].copy()
adata_mut = adata[adata.obs['orig.ident'] == 'pax6_Mutant'].copy()

print(adata_wt.obs.shape)
print(adata_mut.obs.shape)

(6399, 5)
(8582, 5)


In [16]:
import pandas as pd

def get_expr_matrix(adata_subset):
    if hasattr(adata_subset.X, "toarray"):
        return pd.DataFrame(adata_subset.X.toarray(), index=adata_subset.obs_names, columns=adata_subset.var_names)
    else:
        return pd.DataFrame(adata_subset.X, index=adata_subset.obs_names, columns=adata_subset.var_names)

ex_matrix_wt = get_expr_matrix(adata_wt)
ex_matrix_mut = get_expr_matrix(adata_mut)

In [17]:
print("WT Matrix Shape (Cells, Genes):", ex_matrix_wt.shape)
print("Mutant Matrix Shape (Cells, Genes):", ex_matrix_mut.shape)

WT Matrix Shape (Cells, Genes): (6399, 22806)
Mutant Matrix Shape (Cells, Genes): (8582, 22806)


In [18]:
import glob
from arboreto.utils import load_tf_names
from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase

In [19]:
HS_TFS_FNAME = "hs_hgnc_tfs.txt"
MOTIF_ANNOTATIONS_FNAME = "motifs-v9-nr.hgnc-m0.001-o0.0.tbl"
DATABASES_GLOB = "hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather"
tf_names = load_tf_names(HS_TFS_FNAME)
print(f"Loaded {len(tf_names)} transcription factors.")

# 2. Load cisTarget ranking databases
db_fnames = glob.glob(DATABASES_GLOB)
def get_db_name(fname):
    return os.path.splitext(os.path.basename(fname))[0]

dbs = [RankingDatabase(fname=fname, name=get_db_name(fname)) for fname in db_fnames]
print(f"Loaded {len(dbs)} ranking database(s): {[db.name for db in dbs]}")

Loaded 1839 transcription factors.
Loaded 1 ranking database(s): ['hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings']


In [20]:
from arboreto.algo import grnboost2

TypeError: descriptor '__call__' for 'type' objects doesn't apply to a 'property' object

In [21]:
print("Sample expression matrix columns (genes):", list(ex_matrix_wt.columns[:10]))
print("Sample transcription factors:", tf_names[:10])

# Check how many overlap directly
overlap = set(ex_matrix_wt.columns).intersection(set(tf_names))
print(f"Number of direct matching TFs: {len(overlap)}")

Sample expression matrix columns (genes): ['LOC101732307', 'pi16', 'dok1', 'LOC101730410', 'mrps26', 'm1ap', 'loxl3', 'bbc3', 'htra2', 'LOC116408404']
Sample transcription factors: ['HOXA9', 'ZNF8', 'ZNF853', 'NR1H2', 'NR1H3', 'NR1H4', 'NR1I2', 'NR1I3', 'NR2F1', 'NR2F6']
Number of direct matching TFs: 0


In [22]:
upper_genes = [str(g).upper() for g in ex_matrix_wt.columns]
upper_overlap = set(upper_genes).intersection(set(tf_names))
print(f"Overlap with uppercase conversion: {len(upper_overlap)}")

Overlap with uppercase conversion: 1266


In [23]:
# Convert column names to uppercase to match the human TF list
ex_matrix_wt.columns = [str(c).upper() for c in ex_matrix_wt.columns]
ex_matrix_mut.columns = [str(c).upper() for c in ex_matrix_mut.columns]

print("WT columns updated. Ready for GRNBoost2.")
print("Mutant columns updated. Ready for GRNBoost2.")

WT columns updated. Ready for GRNBoost2.
Mutant columns updated. Ready for GRNBoost2.


In [33]:
!pip install "dask==2026.8.0" "distributed==2026.8.0"

  Using cached dask-2026.8.0-py3-none-any.whl.metadata (3.8 kB)
Using cached dask-2026.8.0-py3-none-any.whl (1.5 MB)
  Attempting uninstall: dask
    Found existing installation: dask 2024.2.1
    Uninstalling dask-2024.2.1:
      Successfully uninstalled dask-2024.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-expr 0.5.3 requires dask==2024.2.1, but you have dask 2026.8.0 which is incompatible.


In [34]:
!pip install "dask==2024.2.1" "distributed==2024.2.1" "dask-expr==0.5.3" --force-reinstall

  Using cached dask-2024.2.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached dask_expr-0.5.3-py3-none-any.whl.metadata (2.5 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached partd-1.4.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached toolz-1.1.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached importlib_metadata-9.0.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached locket-1.0.0-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached msgpack-1.2.2-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.3 kB)
  Using cached psutil-7.2.2-cp36-abi3-macosx_11_0_arm64.whl.metadata (22 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached tblib-3.2.2-py3-n

In [24]:
from distributed import LocalCluster, Client
from arboreto.algo import grnboost2

# 1. Start a clean local Dask cluster
cluster = LocalCluster(n_workers=4, threads_per_worker=2, dashboard_address=None)
client = Client(cluster)

# 2. Run GRNBoost2 for PAX6 WT
print("--- Running GRNBoost2 for PAX6 WT ---")
adjacencies_wt = grnboost2(
    ex_matrix_wt, 
    tf_names=tf_names, 
    client_or_address=client,
    verbose=True
)
print(f"Inferred {len(adjacencies_wt):,} regulatory links for WT.")

# 3. Run GRNBoost2 for PAX6 Mutant
print("\n--- Running GRNBoost2 for PAX6 Mutant ---")
adjacencies_mut = grnboost2(
    ex_matrix_mut, 
    tf_names=tf_names, 
    client_or_address=client,
    verbose=True
)
print(f"Inferred {len(adjacencies_mut):,} regulatory links for Mutant.")

# Clean up client
client.close()
cluster.close()

TypeError: descriptor '__call__' for 'type' objects doesn't apply to a 'property' object

In [26]:
import dask.utils
import inspect

# Monkeypatch dask's get_named_args to safely handle Python 3.13 property signatures
old_get_named_args = dask.utils.get_named_args
def safe_get_named_args(func):
    try:
        return old_get_named_args(func)
    except (TypeError, ValueError):
        return []
dask.utils.get_named_args = safe_get_named_args

# Now import and run grnboost2 locally
from arboreto.algo import grnboost2

print("--- Running GRNBoost2 for PAX6 WT (Local Mode) ---")
adjacencies_wt = grnboost2(
    ex_matrix_wt, 
    tf_names=tf_names, 
    client_or_address=None,
    verbose=True
)
print(f"Inferred {len(adjacencies_wt):,} regulatory links for WT.")

print("\n--- Running GRNBoost2 for PAX6 Mutant (Local Mode) ---")
adjacencies_mut = grnboost2(
    ex_matrix_mut, 
    tf_names=tf_names, 
    client_or_address=None,
    verbose=True
)
print(f"Inferred {len(adjacencies_mut):,} regulatory links for Mutant.")

--- Running GRNBoost2 for PAX6 WT (Local Mode) ---
preparing dask client
parsing input
creating dask graph
5 partitions
computing dask graph


/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/client.py:3169: UserWarning: Sending large graph of size 1.10 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
2026-09-22 02:00:20,553 - distributed.protocol.core - CRITICAL - Failed to deserialize
Traceback (most recent call last):
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/protocol/core.py", line 175, in loads
    return msgpack.loads(
           ~~~~~~~~~~~~~^
        frames[0], object_hook=_decode_default, use_list=False, **msgpack_opts
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "msgpack/_unpacker.pyx", line 192, in msgpack._cmsgpack.unpackb
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/protocol/core.py", line 172, in _decode_default
    return pickle.loads(sub_head

shutting down client and local cluster


2026-09-22 02:00:21,466 - distributed.core - ERROR - Exception while handling op terminate
Traceback (most recent call last):
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/core.py", line 970, in _handle_comm
    result = await result
             ^^^^^^^^^^^^
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/scheduler.py", line 4147, in close
    self.stop_services()
    ~~~~~~~~~~~~~~~~~~^^
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/node.py", line 79, in stop_services
    application.stop()
    ~~~~~~~~~~~~~~~~^^
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/bokeh/server/tornado.py", line 761, in stop
    raise RuntimeError("Cannot synchronously wait on a running event loop; use 'await stop_async()'")
RuntimeError: Cannot synchronously wait on a running event loop; use 'await stop_

finished


CancelledError: finalize-e0de464ffb2f1d62c1bb947979c93839

In [27]:
import dask
import dask.utils
import inspect
from distributed import LocalCluster, Client
from arboreto.algo import grnboost2

# 1. Monkeypatch to safely handle Python 3.13 property signatures
old_get_named_args = dask.utils.get_named_args
def safe_get_named_args(func):
    try:
        return old_get_named_args(func)
    except (TypeError, ValueError):
        return []
dask.utils.get_named_args = safe_get_named_args

# 2. Prevent Dask from aggressively killing workers due to memory spikes
dask.config.set({
    'distributed.worker.memory.target': False,
    'distributed.worker.memory.spill': False,
    'distributed.worker.memory.pause': False,
    'distributed.worker.memory.terminate': False
})

# 3. Spin up a single-worker cluster to share memory (prevents duplicating the huge matrix)
print("Configuring memory-safe Dask cluster...")
cluster = LocalCluster(
    n_workers=1,           # 1 Process (crucial for saving RAM)
    threads_per_worker=4,  # 4 Threads (still computes quickly using 4 cores)
    dashboard_address=None
)
client = Client(cluster)
print("Client ready!")

# 4. Run GRNBoost2 for PAX6 WT
print("\n--- Running GRNBoost2 for PAX6 WT ---")
adjacencies_wt = grnboost2(
    ex_matrix_wt, 
    tf_names=tf_names, 
    client_or_address=client,
    verbose=True
)
print(f"Inferred {len(adjacencies_wt):,} regulatory links for WT.")

# 5. Run GRNBoost2 for PAX6 Mutant
print("\n--- Running GRNBoost2 for PAX6 Mutant ---")
adjacencies_mut = grnboost2(
    ex_matrix_mut, 
    tf_names=tf_names, 
    client_or_address=client,
    verbose=True
)
print(f"Inferred {len(adjacencies_mut):,} regulatory links for Mutant.")

# Clean up
client.close()
cluster.close()

Configuring memory-safe Dask cluster...


/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 58876 instead
  warnings.warn(


Client ready!

--- Running GRNBoost2 for PAX6 WT ---
preparing dask client
parsing input
creating dask graph
1 partitions
computing dask graph


/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/client.py:3169: UserWarning: Sending large graph of size 1.10 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
2026-09-22 02:01:20,414 - distributed.protocol.core - CRITICAL - Failed to deserialize
Traceback (most recent call last):
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/protocol/core.py", line 175, in loads
    return msgpack.loads(
           ~~~~~~~~~~~~~^
        frames[0], object_hook=_decode_default, use_list=False, **msgpack_opts
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "msgpack/_unpacker.pyx", line 192, in msgpack._cmsgpack.unpackb
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/protocol/core.py", line 172, in _decode_default
    return pickle.loads(sub_head

not shutting down client, client was created externally
finished


2026-09-22 02:01:20,928 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/comm/tcp.py", line 225, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/worker.py", line 1252, in heartbeat
    response = await retry_operation(
               ^^^^^^^^^^^^^^^^^^^^^^
    ...<14 lines>...
    )
    ^
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/utils_comm.py", line 455, in retry_operation
    return await retry(
           ^^^^

CancelledError: finalize-ed35a174c54dd658ad251d8421e86606

3.13/site-packages/dask/utils.py", line 987, in wrapper
    method.__doc__ = _derived_from(
                     ~~~~~~~~~~~~~^
        original_klass,
        ^^^^^^^^^^^^^^^
    ...<4 lines>...
        inconsistencies=inconsistencies,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/dask/utils.py", line 940, in _derived_from
    method_args = get_named_args(method)
  File "/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/dask/utils.py", line 701, in get_named_args
    s = inspect.signature(func)
  File "/Users/marinmarcillac/miniconda3/lib/python3.13/inspect.py", line 3389, in signature
    return Signature.from_callable(obj, follow_wrapped=follow_wrapped,
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                   globals=globals, locals=locals, eval_str=eval_str)
                                   ^^^^^^^^^^^^^^^^

In [28]:
import dask
import dask.utils
import inspect
import arboreto.algo

# 1. Monkeypatch dask's get_named_args for Python 3.13 property signatures
old_get_named_args = dask.utils.get_named_args
def safe_get_named_args(func):
    try:
        return old_get_named_args(func)
    except (TypeError, ValueError):
        return []
dask.utils.get_named_args = safe_get_named_args

# 2. Monkeypatch Arboreto's client preparation to completely prevent LocalCluster creation
def patched_prepare_client(client_or_address):
    # Return (None, dummy_shutdown) so it runs locally using standard threads
    return None, lambda verbose: None

arboreto.algo._prepare_client = patched_prepare_client

# 3. Now run GRNBoost2 locally and safely for both conditions
from arboreto.algo import grnboost2

print("--- Running GRNBoost2 for PAX6 WT (Safe Local Mode) ---")
adjacencies_wt = grnboost2(
    ex_matrix_wt, 
    tf_names=tf_names, 
    client_or_address=None,
    verbose=True
)
print(f"Inferred {len(adjacencies_wt):,} regulatory links for WT.")

print("\n--- Running GRNBoost2 for PAX6 Mutant (Safe Local Mode) ---")
adjacencies_mut = grnboost2(
    ex_matrix_mut, 
    tf_names=tf_names, 
    client_or_address=None,
    verbose=True
)
print(f"Inferred {len(adjacencies_mut):,} regulatory links for Mutant.")

--- Running GRNBoost2 for PAX6 WT (Safe Local Mode) ---
preparing dask client
parsing input
creating dask graph
finished


AssertionError: client is required